<div align="center">

# OpenEnv: ESG Compliance & Sustainability

### *RL Training with GRPO + Unsloth*

---

**Train an LLM to act as a corporate ESG strategist using verifiable environment rewards.**

[![GitHub](https://img.shields.io/badge/GitHub-TharunBabu--05%2FOPEN--ENV-blue?logo=github)](https://github.com/TharunBabu-05/OPEN-ENV)
[![License](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)

</div>

---

## What You'll Do

<table>
<tr>
<td width="50%">

**Part 1: Setup**
- GPU check & dependency install
- Clone repo & validate imports
- Smoke test the full pipeline

</td>
<td width="50%">

**Part 2: Train & Evaluate**
- Generate training dataset
- Run GRPO training (~45 min on T4)
- Benchmark & visualize results
- Push trained model to HuggingFace

</td>
</tr>
</table>

## Architecture

```
+---------------------------------------------------------+
|  TRAINING CODE (this notebook)                          |
|                                                         |
|  dataset_builder.py  --> prompts (JSONL)                |
|  train_rl.py         --> GRPOTrainer (TRL + Unsloth)    |
|  reward_functions.py --> 4 verifiable reward signals    |
|                                                         |
+----------------------------+----------------------------+
                             |
                             v
+----------------------------+----------------------------+
|  ESG ENVIRONMENT (env.py)                               |
|                                                         |
|  reset() --> Observation (17 fields)                    |
|  step(action) --> (obs, reward, done, truncated, info)  |
|  state() --> current observation                        |
|                                                         |
|  9 actions | 3 tasks | deterministic grading            |
+---------------------------------------------------------+
```

> **Runtime:** ~50 min total on Colab T4 GPU (free tier)

---

# Part 1: Setup

## Cell 1 -- GPU Check

In [ ]:
# Cell 1 -- GPU Check
import subprocess, sys, os

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print('WARNING: No GPU detected!')
    print('Go to: Runtime > Change runtime type > T4 GPU')
    print('Then re-run this cell.')
else:
    print(result.stdout)

import torch
assert torch.cuda.is_available(), 'STOP: Enable T4 GPU in Runtime menu first!'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print('\nReady to train!')

## Cell 2 -- Install Dependencies

Installs Unsloth (fast 4-bit LoRA), TRL (GRPO trainer), and project dependencies.

In [ ]:
# Cell 2 -- Install dependencies (~3-5 min)
!pip install -q pydantic openai pyyaml datasets
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q trl peft accelerate bitsandbytes
print('\nAll dependencies installed.')

## Cell 3 -- Clone Repository & Validate

In [ ]:
# Cell 3 -- Clone repo and validate
REPO_URL = 'https://github.com/TharunBabu-05/OPEN-ENV.git'
REPO_DIR = '/content/open_ENV'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print('Repository already present -- pulled latest.')

%cd {REPO_DIR}

# Validate imports
from env import ESGEnvironment
from tasks import TASKS, grade_task
from models import Observation, Action

print('\n' + '='*60)
print('  ENVIRONMENT VALIDATED')
print('='*60)
print(f'  Tasks: {list(TASKS.keys())}')
for tid, t in TASKS.items():
    print(f'    {tid:30s} | {t.difficulty:6s} | {t.max_steps} steps | ${t.initial_budget:,.0f}')
print('='*60)

## Cell 4 -- Smoke Test

Validates the full pipeline **without** loading a model or using GPU:
- Dataset generation
- Reward functions (format, anti-cheat, composite)
- Environment stepping

In [ ]:
# Cell 4 -- Smoke test (no model needed)
!python train_rl.py --smoke_test
print('\nIf you see SMOKE TEST PASSED above, proceed to Part 2.')

---

# Part 2: Train & Evaluate

## Cell 5 -- Generate Training Dataset

Rolls out the ESG environment with a heuristic policy to create diverse `(observation, prompt)` training pairs.

In [ ]:
# Cell 5 -- Generate dataset
!python dataset_builder.py --episodes 5 --output data/esg_prompts.jsonl

import json
with open('data/esg_prompts.jsonl') as f:
    lines = f.readlines()
sample = json.loads(lines[0])
print(f'\nDataset: {len(lines)} prompts')
print(f'Tasks:   {set(json.loads(l)["task_id"] for l in lines)}')
print(f'Sample:  task={sample["task_id"]}, step={sample["step"]}')

## Cell 6 -- GRPO Training

This is the main training cell. Uses `train_config_fast.yaml`:
- **Model:** `unsloth/Qwen2.5-0.5B-Instruct` (small, fast)
- **Steps:** 150 (~45 min on T4)
- **Task:** `basic_compliance` (easiest, guarantees reward signal)

| Config | Value | Why |
|--------|-------|-----|
| LoRA rank | 8 | Fewer params = faster training |
| Batch size | 1 | Fits T4 16GB VRAM |
| GRPO rollouts | 4 | Balance exploration vs speed |
| Temperature | 0.7 | Encourages diverse actions |

In [ ]:
# Cell 6 -- GRPO training (~45 min on T4)
!python train_rl.py \
    --config train_config_fast.yaml \
    --output_dir outputs/esg_rl_trained

import os
adapter = 'outputs/esg_rl_trained/lora_adapter'
if os.path.exists(adapter):
    print('\n' + '='*60)
    print('  TRAINING COMPLETE')
    print('='*60)
    print('  Adapter files:', os.listdir(adapter))
    print('='*60)
else:
    print('ERROR: Adapter not found -- check training logs above.')

## Cell 7 -- Benchmark & Visualize

Compares the **heuristic baseline** against the **trained model** across all 3 task difficulties.

In [ ]:
# Cell 7 -- Benchmark baseline vs trained model
import os, json

# Heuristic baseline
!python benchmark.py --mode heuristic --seeds 42 43 44 --output results/baseline_heuristic.json

# Trained model (if adapter exists)
adapter = 'outputs/esg_rl_trained/lora_adapter'
if os.path.exists(adapter):
    !python benchmark.py --mode llm --model_path {adapter} --seeds 42 43 44 --output results/trained.json
    !python plot_results.py --baseline results/baseline_heuristic.json --trained results/trained.json --output_dir results
else:
    !python plot_results.py --baseline results/baseline_heuristic.json --output_dir results

# Print scores
print('\n' + '='*60)
print('  BENCHMARK RESULTS')
print('='*60)
baseline = json.load(open('results/baseline_heuristic.json'))
print(f'  Heuristic baseline: {baseline["overall_mean_score"]:.3f}')
if os.path.exists('results/trained.json'):
    trained = json.load(open('results/trained.json'))
    delta = trained['overall_mean_score'] - baseline['overall_mean_score']
    print(f'  Trained model:      {trained["overall_mean_score"]:.3f} ({delta:+.3f})')
print('='*60)

# Display plots
from IPython.display import Image, display
for plot in ['score_comparison.png', 'reward_history.png', 'esg_metrics.png']:
    path = f'results/{plot}'
    if os.path.exists(path):
        print(f'\n--- {plot} ---')
        display(Image(path))

## Cell 8 -- Push to HuggingFace Hub

Uploads the trained LoRA adapter and result plots to HuggingFace.

> **Before running:** Replace `HF_TOKEN` below with your write-access token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

In [ ]:
# Cell 8 -- Push model + results to HuggingFace
#
# >>> FILL IN YOUR TOKEN BELOW <<<
HF_TOKEN    = 'hf_REPLACE_ME'
HF_USERNAME = 'tharun5054'
MODEL_REPO  = f'{HF_USERNAME}/esg-rl-agent-grpo'

assert HF_TOKEN != 'hf_REPLACE_ME', 'Set your HF_TOKEN above first!'

from huggingface_hub import login, HfApi, create_repo
login(token=HF_TOKEN)

create_repo(MODEL_REPO, exist_ok=True, private=False)

api = HfApi(token=HF_TOKEN)
api.upload_folder(
    folder_path='outputs/esg_rl_trained/lora_adapter',
    repo_id=MODEL_REPO,
    repo_type='model',
)
print(f'Model pushed -> https://huggingface.co/{MODEL_REPO}')

# Push result plots as evidence
import os
for plot in ['score_comparison.png', 'reward_history.png', 'esg_metrics.png']:
    if os.path.exists(f'results/{plot}'):
        api.upload_file(
            path_or_fileobj=f'results/{plot}',
            path_in_repo=f'results/{plot}',
            repo_id=MODEL_REPO,
            repo_type='model',
        )
print(f'\nDone! Visit: https://huggingface.co/{MODEL_REPO}')

---

# Summary

<table>
<tr><th>What</th><th>How</th></tr>
<tr><td><b>Environment</b></td><td>ESGEnvironment -- 9 actions, 17-field observations, 3 difficulty levels</td></tr>
<tr><td><b>Reward</b></td><td>4 independent verifiable signals (no LLM-as-judge)</td></tr>
<tr><td><b>Training</b></td><td>GRPO via TRL + Unsloth (4-bit LoRA)</td></tr>
<tr><td><b>Anti-cheat</b></td><td>NO_ACTION spam penalty, format compliance check, budget abuse detection</td></tr>
<tr><td><b>Evidence</b></td><td>Deterministic benchmarks + comparison plots in results/</td></tr>
</table>

### Links

- [GitHub Repository](https://github.com/TharunBabu-05/OPEN-ENV)
- [HuggingFace Space](https://huggingface.co/spaces/tharun5054/esg-compliance-env)
- [OpenEnv Framework](https://github.com/meta-pytorch/OpenEnv)